# Customer Shopping Analytics — Data Preparation & EDA

Cleans the raw customer shopping dataset and prepares it for SQL analysis and the Power BI
dashboard. Includes explicit validation checks (nulls, duplicates, value ranges) after each
transformation, rather than assuming a step worked.


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


## 1. Load raw data

In [3]:
df = pd.read_csv("customer_shopping_behavior.csv")
print(df.shape)
df.head()


(3900, 18)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

## 2. Check for missing values

In [5]:
df.isnull().sum()


,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


Only `Review Rating` has missing values (37 rows, <1%). Impute using the **category median**
rather than a single global median — ratings can differ meaningfully by product category, so this
preserves that signal instead of flattening it.


In [6]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
assert df['Review Rating'].isnull().sum() == 0, "Review Rating still has nulls after imputation"
print("Review Rating nulls remaining:", df['Review Rating'].isnull().sum())


Review Rating nulls remaining: 0


## 3. Check for duplicate rows

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Customer IDs:", df['Customer ID'].duplicated().sum())


Exact duplicate rows: 0
Duplicate Customer IDs: 0


## 4. Validate value ranges

In [8]:
assert df['Age'].between(0, 120).all(), "Age has implausible values"
assert (df['Purchase Amount (USD)'] > 0).all(), "Purchase Amount has non-positive values"
assert df['Review Rating'].between(1, 5).all(), "Review Rating outside expected 1-5 range"

print("Age range:", df['Age'].min(), "-", df['Age'].max())
print("Purchase Amount range:", df['Purchase Amount (USD)'].min(), "-", df['Purchase Amount (USD)'].max())
print("Review Rating range:", df['Review Rating'].min(), "-", df['Review Rating'].max())
print("All range checks passed.")


Age range: 18 - 70
Purchase Amount range: 20 - 100
Review Rating range: 2.5 - 5.0
All range checks passed.


## 5. Standardize column names

In [9]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})
df.columns.tolist()


['customer_id',
 'age',
 'gender',
 'item_purchased',
 'category',
 'purchase_amount',
 'location',
 'size',
 'color',
 'season',
 'review_rating',
 'subscription_status',
 'shipping_type',
 'discount_applied',
 'promo_code_used',
 'previous_purchases',
 'payment_method',
 'frequency_of_purchases']

## 6. Create `age_group`

Using a quartile split (`pd.qcut`) so each group has a roughly equal number of customers. These are
**statistical quartile bins based on this dataset's age distribution, not standard demographic
age brackets** — the exact cutoffs are printed below so the labels have a precise, defensible
definition rather than an implied one.


In [10]:
labels = ['Young Adults', 'Adult', 'Middle-aged', 'Senior']
df['age_group'], bins = pd.qcut(df['age'], q=4, labels=labels, retbins=True)

print("Age group boundaries:")
for i, label in enumerate(labels):
    print(f"  {label}: {bins[i]:.0f} - {bins[i+1]:.0f}")

df[['age', 'age_group']].head(10)


Age group boundaries:
  Young Adults: 18 - 31
  Adult: 31 - 44
  Middle-aged: 44 - 57
  Senior: 57 - 70


,age,age_group
0,55,Middle-aged
1,19,Young Adults
2,50,Middle-aged
3,21,Young Adults
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adults
8,26,Young Adults
9,57,Middle-aged


## 7. Create `purchase_frequency_days`

**Corrected mapping** — the first version used `'Bi-weekly'` and `'Every 3 months'`, which didn't
match the dataset's actual casing (`'Bi-Weekly'`, `'Every 3 Months'`). That silently NaN'd 1,131 rows
(29% of the data) with no error raised. Verified against the real category values below before
mapping, and asserted zero nulls after.


In [11]:
print("Actual values in Frequency of Purchases:")
print(df['frequency_of_purchases'].value_counts())


Actual values in Frequency of Purchases:
frequency_of_purchases
Every 3 Months    584
Annually          572
Quarterly         563
Monthly           553
Bi-Weekly         547
Fortnightly       542
Weekly            539
Name: count, dtype: int64


In [12]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

null_count = df['purchase_frequency_days'].isnull().sum()
assert null_count == 0, f"purchase_frequency_days has {null_count} unmapped rows — check for casing mismatches"
print("Unmapped rows:", null_count)

df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)


Unmapped rows: 0


,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


## 8. Check for redundant columns

`discount_applied` and `promo_code_used` look like they might be the same signal — verify before
dropping either one, rather than assuming.


In [13]:
match_rate = (df['discount_applied'] == df['promo_code_used']).mean()
print(f"discount_applied and promo_code_used match on {match_rate:.1%} of rows")

if match_rate == 1.0:
    df = df.drop('promo_code_used', axis=1)
    print("Dropped promo_code_used (100% identical to discount_applied)")
else:
    print("Columns differ — keeping both, do not drop")


discount_applied and promo_code_used match on 100.0% of rows
Dropped promo_code_used (100% identical to discount_applied)


In [14]:
df.columns.tolist()


['customer_id',
 'age',
 'gender',
 'item_purchased',
 'category',
 'purchase_amount',
 'location',
 'size',
 'color',
 'season',
 'review_rating',
 'subscription_status',
 'shipping_type',
 'discount_applied',
 'previous_purchases',
 'payment_method',
 'frequency_of_purchases',
 'age_group',
 'purchase_frequency_days']

## 9. Final validation before export

In [15]:
print("Final shape:", df.shape)
print()
print("Remaining nulls:\n", df.isnull().sum().sum(), "total")
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("age_group value counts:\n", df['age_group'].value_counts())
print()
print("purchase_frequency_days dtype:", df['purchase_frequency_days'].dtype)


Final shape: (3900, 19)

Remaining nulls:
 0 total

Duplicate rows: 0

age_group value counts:
 age_group
Young Adults    1028
Middle-aged      986
Senior           944
Adult            942
Name: count, dtype: int64

purchase_frequency_days dtype: int64


## 10. Export cleaned data

In [16]:
df.to_csv("cleaned_customer_shopping.csv", index=False)
print("Exported: cleaned_customer_shopping.csv")
print(f"Final shape: {df.shape}")


Exported: cleaned_customer_shopping.csv
Final shape: (3900, 19)
